In [14]:
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR / "src"))

env_file = ROOT_DIR / ".env"
if env_file.is_file():
    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

from eoa import (
    ExplanationLayer,
    SpecializedAgent,
    SQLiteSessionStore,
)

In [15]:
incident_402 = {
    "prediction": {
        "model_name": "XGBoost_CoolerCondition_v2",
        "target_variable": "Condição do Resfriador Hidráulico",
        "predicted_value": "Eficiência Reduzida (Nível Crítico)",
        "confidence_or_probability": 0.86,
        "status_label": "Operação Degradada / Alerta Crítico",
    },
    "xai": {
        "method": "SHAP",
        "feature_attributions": {
            "PS1_Pressao_Entrada": 0.54,
            "EPS1_Potencia_Motor": 0.32,
            "TS2_Temp_Trocador": 0.14,
        },
    },
    "context": {
        "variables": {
            "PS1_Pressao_Entrada": "Bar, faixa normal de operação [2.0, 4.0] bar",
            "EPS1_Potencia_Motor": "kW, faixa normal de operação [1.5, 2.8] kW",
        },
        "thresholds": {"PS1_max": "4.0 bar", "EPS1_max": "3.0 kW"},
        "documents": {
            "POP_Resfriamento_04": "Seção 4: Se PS1 exceder 4.0 bar acompanhado por EPS1 > 3.0 kW, há provável cavitação no filtro de sucção. Reduzir carga hidráulica imediatamente."
        },
    },
}

In [16]:
store = SQLiteSessionStore(db_path=ROOT_DIR / "examples" / "live_gemini_sessions.db")
chat_layer = ExplanationLayer.default_layer(session_store=store)
session_id = "sessao_real_turbina_01"
chat_layer.clear_session(session_id)

In [17]:
res1 = chat_layer.explain(
    prediction=incident_402["prediction"],
    xai=incident_402["xai"],
    context=incident_402["context"],
    user_query="Resuma o problema detectado no resfriador.",
    session_id=session_id,
)
print("[TURNO 1]:\n", res1.narrative)

[TURNO 1]:
 O sistema preditivo (XGBoost_CoolerCondition_v2) detectou uma condição crítica de operação degradada no resfriador hidráulico ("Eficiência Reduzida"), com um nível de risco e confiança de 86,00%. Esta indicação aponta para uma perda significativa na capacidade de troca térmica ou no desempenho geral do sistema de refrigeração hidráulica.

O modelo chegou a essa conclusão impulsionado principalmente por três variáveis monitoradas: a pressão de entrada `PS1_Pressao_Entrada`, que apresentou o maior peso de atribuição positiva (+0.5400), seguida pela potência do motor `EPS1_Potencia_Motor` (+0.3200) e pela temperatura no trocador `TS2_Temp_Trocador` (+0.1400). Todas essas características convergiram positivamente para o estado de falha crítica. Conforme os parâmetros operacionais de domínio (CMA), a faixa normal de operação da pressão `PS1` é de [2.0, 4.0] bar (com limite máximo de 4.0 bar) e a potência `EPS1` opera normalmente entre [1.5, 2.8] kW (com máximo de 3.0 kW). Valore

In [18]:
res2 = chat_layer.explain(
    user_query="Se não pudermos parar a bomba agora, qual o risco para o motor elétrico (EPS1)?",
    session_id=session_id,
)
print("\n[TURNO 2 - FOLLOW-UP]:\n", res2.narrative)
print(f"\nScore Turno 2: {res2.validation_score}/10.0 | Mensagens no banco: {len(chat_layer.get_history(session_id))}")


[TURNO 2 - FOLLOW-UP]:
 Caso a bomba não possa ser interrompida imediatamente, a manutenção da operação sob as condições atuais expõe o motor elétrico (`EPS1`) a um risco elevado de sobrecarga térmica e mecânica. Como o modelo indica uma pressão de entrada (`PS1`) elevada e uma potência do motor (`EPS1`) atuando como principais vetores do estado crítico de eficiência reduzida, operar o sistema além da faixa normal (com potência limite de 3.0 kW e pressão de 4.0 bar) pode gerar desgaste acelerado dos enrolamentos do motor e falhas por esforço excessivo no circuito hidráulico.

Para mitigar o risco ao motor elétrico na impossibilidade de paragem total, deve-se verificar prioritariamente se a potência (`EPS1`) está excedendo o limite máximo de 3.0 kW em conjunto com a pressão (`PS1`) acima de 4.0 bar, cenário que caracteriza cavitação no filtro de sucção conforme o procedimento `POP_Resfriamento_04`. Como ação paliativa imediata, proceda com a redução controlada da carga hidráulica para 

In [19]:
res3 = await chat_layer.aexplain(
    user_query="Me explica as variáveis que estamos tratando e o que elas representam, você tem alguma tool associada?",
    session_id=session_id,
)

print(f"\n[TURNO 3 - FOLLOW-UP (Async)] (Score: {res3.validation_score}/10.0):\n", res3.narrative)


[TURNO 3 - FOLLOW-UP (Async)] (Score: 10.0/10.0):
 As variáveis monitoradas no sistema que estão influenciando o estado crítico do resfriador hidráulico são a `EPS1_Potencia_Motor`, a `PS1_Pressao_Entrada` e a `TS2_Temp_Trocador`. A potência do motor (`EPS1_Potencia_Motor`) representa a carga e o esforço exigidos do motor elétrico do equipamento EPS1, medida em quilowatts (kW), com faixa normal de operação entre 1.5 e 2.8 kW e limite máximo de 3.0 kW. Já a pressão de entrada (`PS1_Pressao_Entrada`) reflete a pressão no ponto de admissão do equipamento PS1, medida em bar, cuja faixa normal situa-se entre 2.0 e 4.0 bar, com limite máximo também em 4.0 bar. Por fim, a temperatura no trocador (`TS2_Temp_Trocador`) está associada às condições térmicas do processo de troca de calor do sistema.

Em relação à sua pergunta sobre ferramentas associadas, os artefatos atuais indicam que **não há ferramentas associadas** definidas no contexto de domínio (CMA) para este cenário.

Para mitigar o pro

In [20]:
chat_layer.override_agent_prompt(
    name="explainer",
    prompt="""
Responda de forma extremamente objetiva e técnica à dúvida do operador, sem rodeios e sem encher linguiça.

Dúvida do operador:
{user_query}
""",
)

chat_layer.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Inline (str)
validator          Quality Gate Auditor     0        Built-in


In [21]:
res4 = chat_layer.explain(
    user_query="O que é um motor de pressão?",
    session_id=session_id,
)

print(f"\n[TURNO 4 - Pós reescrita no prompt] (Score: {res4.validation_score}/10.0):\n", res4.narrative)


[TURNO 4 - Pós reescrita no prompt] (Score: 10.0/10.0):
 Um **motor de pressão** (comumente chamado de motor pneumático) é um atuador que converte energia potencial de um fluido sob pressão (ar comprimido) em **energia mecânica rotativa**. 

Sua operação baseia-se na expansão do gás no interior de câmaras vedadas, acionando rotores ou pistões acoplados a um eixo de saída.


In [22]:
chat_layer.reset_agent_prompt("explainer")


def soma_fake(a: int, b: int) -> int:
    """Multiplica os fatores a e b simulando uma soma fake customizada."""
    return a * b


matematico = SpecializedAgent(
    name="math",
    role="Matemática Fake",
    prompt="""
Você é um especialista em matemática fake.
A ferramenta soma_fake executou e calculou o seguinte resultado numérico: {tool_outputs}
Explique o resultado obtido e o conceito de cálculo fake para a dúvida: {user_query}
""",
)

matematico.register_tool(soma_fake)
chat_layer.register_agent(matematico)

chat_layer.inspect()

EOA Framework - Active Explanation Layer Configuration
Agent              Role                     Tools    Prompt Source
------------------------------------------------------------------------------
context_manager    Context Manager          0        Built-in
artifacts          Artifacts Analyst        0        Built-in
explainer          Explainer                0        Built-in
validator          Quality Gate Auditor     0        Built-in
math               Matemática Fake          1        Inline (str)


In [23]:
res_com_tool = chat_layer.explain(
    user_query="Faça uma soma fake de 10 e 20 e apresente o resultado.",
    session_id=session_id,
)
math_step_a = next(s for s in res_com_tool.execution_trace if s.agent_name == "math")
print("=== CASO A: Pergunta Matemática ('soma fake de 10 e 20') ===")
print("Trace do Agente Math:", math_step_a.summary)
print("Resposta:\n", res_com_tool.narrative)

print("\n" + "=" * 75 + "\n")

res_sem_tool = chat_layer.explain(
    user_query="Explique brevemente o que é cavitação no filtro de sucção segundo o procedimento operacional.",
    session_id=session_id,
)
math_step_b = next(s for s in res_sem_tool.execution_trace if s.agent_name == "math")
print("=== CASO B: Pergunta Conceitual ('o que é cavitação') ===")
print("Trace do Agente Math:", math_step_b.summary)
print("Resposta:\n", res_sem_tool.narrative)

=== CASO A: Pergunta Matemática ('soma fake de 10 e 20') ===
Trace do Agente Math: Specialist 'math' evaluated domain factors (tools executed: [200]).
Resposta:
 Como especialista encarregado da operação e dos procedimentos técnicos do resfriador hidráulico, devo esclarecer que não possuo ferramentas associadas ou diretrizes em nossos manuais operacionais (CMA) para realizar cálculos matemáticos fictícios ou operações de "soma fake". Meu escopo de atuação é estritamente voltado para o monitoramento de variáveis físicas como a potência do motor (`EPS1_Potencia_Motor`), pressão de entrada (`PS1_Pressao_Entrada`) e temperatura do trocador (`TS2_Temp_Trocador`), além da aplicação de procedimentos operacionais padrão, como o `POP_Resfriamento_04` para mitigação de cavitação. 

Caso tenha dúvidas técnicas relacionadas ao comportamento do sistema hidráulico, aos limites operacionais ou às diretrizes de mitigação de falhas, estou à disposição para auxiliar com base rigorosa nos artefatos e nor

In [24]:
history = chat_layer.get_history(session_id=session_id)

for msg in history:
    print(f"[{msg.role.upper()}]: {msg.content}\n" + "-" * 60)

[USER]: Resuma o problema detectado no resfriador.
------------------------------------------------------------
[ASSISTANT]: O sistema preditivo (XGBoost_CoolerCondition_v2) detectou uma condição crítica de operação degradada no resfriador hidráulico ("Eficiência Reduzida"), com um nível de risco e confiança de 86,00%. Esta indicação aponta para uma perda significativa na capacidade de troca térmica ou no desempenho geral do sistema de refrigeração hidráulica.

O modelo chegou a essa conclusão impulsionado principalmente por três variáveis monitoradas: a pressão de entrada `PS1_Pressao_Entrada`, que apresentou o maior peso de atribuição positiva (+0.5400), seguida pela potência do motor `EPS1_Potencia_Motor` (+0.3200) e pela temperatura no trocador `TS2_Temp_Trocador` (+0.1400). Todas essas características convergiram positivamente para o estado de falha crítica. Conforme os parâmetros operacionais de domínio (CMA), a faixa normal de operação da pressão `PS1` é de [2.0, 4.0] bar (com l

In [26]:
# Modo Zero-ML + Hidratação Progressiva de Contexto via Tools no CMA
# (Sem modelo de ML nem XAI: o ArtifactsAgent economiza 100% dos tokens pulando o LLM,
#  e o CMA usa tools sob demanda acumulando documentos na sessão!)

zero_ml_layer = ExplanationLayer.default_layer(session_store=store)
zero_ml_session = "sessao_zero_ml_progressiva"
zero_ml_layer.clear_session(zero_ml_session)

def buscar_manual_valvula(valvula_id: str) -> str:
    """Busca o procedimento operacional padrão de uma válvula hidráulica específica."""
    return f"Manual {valvula_id}: Pressão máxima de alívio é 4.0 bar. Em caso de bloqueio, abrir bypass manual."

cma_agent = zero_ml_layer.get_agent("context_manager")
cma_agent.register_tool(buscar_manual_valvula)

res_zero = zero_ml_layer.explain(
    user_query="Qual o procedimento operacional no manual da válvula V-102 em caso de bloqueio?",
    session_id=zero_ml_session,
)

aa_step = next(s for s in res_zero.execution_trace if s.agent_name == "artifacts")
print("Trace do ArtifactsAgent (Zero-ML):", aa_step.summary)
print("Documentos hidratados no AgentState.context:", zero_ml_layer.last_state.context.documents)
print("\nResposta Zero-ML Validada:\n", res_zero.narrative)

Trace do ArtifactsAgent (Zero-ML): Zero ML mode: Skipped LLM evaluation without prediction.
Documentos hidratados no AgentState.context: {'tool_context_1': 'Manual V-102: Pressão máxima de alívio é 4.0 bar. Em caso de bloqueio, abrir bypass manual.'}

Resposta Zero-ML Validada:
 Com base nas diretrizes operacionais e no contexto de domínio da válvula V-102 (que possui limite máximo de pressão de alívio de 4,0 bar), o procedimento exigido pelo manual em caso de bloqueio da válvula V-102 é abrir o bypass manual.


In [28]:
state = chat_layer.last_state

print("=== 1. ARTEFATOS EM MEMÓRIA NO AGENTSTATE ===")
print("Prediction:", state.prediction.model_dump() if state.prediction else None)
print("Top Features XAI:", state.xai.top_features() if state.xai else None)
print("Context Variables:", list(state.context.variables.keys()) if state.context else None)
print("Context Documents:", list(state.context.documents.keys()) if state.context else None)

print("\n=== 2. BLACKBOARD INTERMEDIÁRIO DOS AGENTES ===")
print("Resumo CMA (context_summary):", (state.context_summary or "")[:160], "...")
print("Resumo AA (artifacts_summary):", (state.artifacts_summary or "")[:160], "...")
print("Especialistas (specialist_insights):", list(state.specialist_insights.keys()))

print("\n=== 3. RASTRO DE EXECUÇÃO E LATÊNCIA (EXECUTION TRACE) ===")
for step in res_sem_tool.execution_trace:
    print(f"  • {step.agent_name:<16} | {step.execution_time_ms:>8.1f} ms | {step.status:<10} | {step.summary}")

print("\n=== 4. AUDITORIAS GRAVADAS NO SQLITE (VALIDATION_LOGS) ===")
for log in store.get_validation_logs(session_id):
    print(f"  • [{log['timestamp']}] Score: {log['score']}/10.0 | Passed: {bool(log['passed'])}")

=== 1. ARTEFATOS EM MEMÓRIA NO AGENTSTATE ===
Prediction: {'target_variable': 'Condição do Resfriador Hidráulico', 'predicted_value': 'Eficiência Reduzida (Nível Crítico)', 'model_name': 'XGBoost_CoolerCondition_v2', 'confidence_or_probability': 0.86, 'status_label': 'Operação Degradada / Alerta Crítico', 'metadata': {}}
Top Features XAI: [('PS1_Pressao_Entrada', 0.54), ('EPS1_Potencia_Motor', 0.32), ('TS2_Temp_Trocador', 0.14)]
Context Variables: ['PS1_Pressao_Entrada', 'EPS1_Potencia_Motor']
Context Documents: ['POP_Resfriamento_04']

=== 2. BLACKBOARD INTERMEDIÁRIO DOS AGENTES ===
Resumo CMA (context_summary): - `Key Variables`:
  - `EPS1_Potencia_Motor`: Motor power, kW, normal operating range [1.5, 2.8] kW, max threshold 3.0 kW
  - `PS1_Pressao_Entrada`: Inlet press ...
Resumo AA (artifacts_summary): - `Outcome`: 
  - Target: Condição do Resfriador Hidráulico
  - Predicted Value: Eficiência Reduzida (Nível Crítico)
  - Status: Operação Degradada / Alerta Crí ...
Especialistas (spe